# FDM-PINN plate bending under HYDROSTATIC (linearly varying) loading

In [1]:
"""
FDM-PINN (Finite-Difference-Method PINN) for Kirchhoff-Love plate bending
under HYDROSTATIC (linearly varying) loading, ALL FOUR EDGES CLAMPED /
FIXED -- referred to throughout this script as the "FFFF" case
(Fixed-Fixed-Fixed-Fixed), per the user's naming convention.

    q(x, y) = q0 * (x / a)        -- zero at x=0, rising linearly to q0 at
                                      x=a, UNIFORM across y.

Note "FFFF" here is the CLASSICAL "clamped" plate boundary condition
(w = 0 AND dw/dn = 0 on every edge) -- not the "F = free edge" convention
sometimes used elsewhere in plate literature; here "F" = "Fixed".

This is the FFFF sibling of fdm_pinn_hydrostatic_hardBC.py, following the
same pattern already used for fdm_pinn_sinusoidal_hardBC_FFFF.py and
fdm_pinn_uniform_hardBC_FFFF.py (which in turn followed
vanilla_pinn_hydrostatic_hardBC_FFFF.py's Rayleigh-Ritz reference
machinery). The FD-stencil derivative machinery, network architecture,
training loop, and output artifact schema are all UNCHANGED from the
SS-SS-SS-SS hydrostatic-load script. What changes is the boundary
condition (hence the hard-BC envelope and the reference solution) and the
removal of the moment-based soft loss.

-----------------------------------------------------------------------------
ANALYTICAL / REFERENCE SOLUTION -- RAYLEIGH-RITZ, NOT A NAVIER SERIES
-----------------------------------------------------------------------------
The SS-SS-SS-SS hydrostatic script used the classical Navier double
Fourier series (Timoshenko & Woinowsky-Krieger, Sec. 28) -- valid because
a simply-supported edge's natural BC (w=0, M=0) is satisfied term-by-term
by every sin(m pi x/a) sin(n pi y/b) mode individually.

A clamped edge needs w=0 AND dw/dn=0 simultaneously, which no single
sin/cos term satisfies on opposite edges together -- so, exactly as for
the other two FFFF ports (sinusoidal and uniform load), there is no
elementary closed-form Navier series here either. This script again uses
the RAYLEIGH-RITZ (energy/Galerkin) method with the clamped-admissible
basis

    phi_i(xi) = 1 - cos(2*i*pi*xi),   i = 1, 2, 3, ...

which satisfies phi_i(0)=phi_i(1)=0 AND phi_i'(0)=phi_i'(1)=0 exactly (see
vanilla_pinn_hydrostatic_hardBC_FFFF.py's docstring for the full stiffness-
matrix derivation -- K is assembled exactly via closed-form 1-D integrals
and Kronecker products, no quadrature needed, and is IDENTICAL in
structure here since K depends only on the clamped basis and Poisson's
ratio, never on the load).

The only thing that changes for THIS load case is the load-projection
integral. For the hydrostatic ramp q(x,y) = q0*(x/a), the needed 1-D
integrals are:

    I_i^x = Int_0^1 xi * phi_i(xi) dxi = 0.5     (ramp direction, ALL i --
                                                    see the same derivation
                                                    used in the vanilla
                                                    hydrostatic-FFFF script)
    I_j^y = Int_0^1 1  * phi_j(eta) deta = 1      (uniform-in-y direction,
                                                    ALL j -- same as the
                                                    uniform-load-FFFF script)

Because the load separates as q0*(x/a)*1, the 2-D load vector is

    F_mn = q0 * I_m^x * I_n^y = q0 * 0.5 * 1 = 0.5*q0

a CONSTANT right-hand side (same value for every single (m,n) pair,
exactly half of the uniform-load-FFFF script's F=q0), so K @ c_flat =
F_flat is solved once via np.linalg.solve.

VALIDATION (carried over from the SS hydrostatic script's own validation
logic, and RE-VERIFIED here for the clamped case specifically): the
argument that q0*x/a = q0/2 + (q0/2)*(2x/a - 1) splits into a symmetric
half (uniform q0/2) plus a half that is ANTISYMMETRIC about x=a/2 relies
only on the PDE being linear and the boundary conditions being symmetric
under x -> a-x -- BOTH of which hold for the clamped (FFFF) case exactly
as they did for SS-SS-SS-SS. So the same exact-half relationship must
still hold:

    hydrostatic_FFFF center deflection == EXACTLY HALF of
    uniform_FFFF center deflection

Checked numerically at M=N=18: uniform-FFFF center = 0.00126524 (nd),
hydrostatic-FFFF center = 0.00063262 (nd) -- exactly half, to the
precision of the Ritz solve. This is a genuine cross-check between two
independently-assembled Ritz systems (different load vectors, same K),
which is reassuring even though (like the sinusoidal-FFFF case) no
external textbook benchmark for the clamped+hydrostatic combination was
located. RITZ_M_TERMS=RITZ_N_TERMS=18 (same truncation used in the other
two FFFF FDM-PINN scripts) is comfortably converged: the center deflection
changes by <0.01% going from (18,18) to (24,24).

-----------------------------------------------------------------------------
WHAT ELSE CHANGED FROM THE SS-SS-SS-SS HYDROSTATIC FDM-PINN SCRIPT
-----------------------------------------------------------------------------
1) HARD-BC ENVELOPE (squared, to also zero the slope):
       envelope = [ (x/a)*(1-x/a) ]^2 * [ (y/b)*(1-y/b) ]^2
   instead of the SS case's un-squared xi*(1-xi)*eta*(1-eta). Squaring
   each factor zeros BOTH the function value AND its normal derivative at
   every edge.

2) MOMENT LOSS REMOVED FROM TRAINING: Mx=0 / My=0 on the edges was the
   correct natural BC for a SIMPLY-SUPPORTED edge; for a CLAMPED edge Mx,
   My are generally NONZERO there (the edge moment is exactly what
   resists rotation), so enforcing zero would now be physically wrong.
   It's computed only as a diagnostic. A new bc_slope_loss diagnostic is
   added (should be ~0, confirming the squared envelope truly zeros the
   slope).

   Total training loss for this script is therefore PDE residual only.

3) OUT_DIR now keys on MODEL_NAME, LOADING_CASE, *and* BC_CASE (extending
   the same naming-collision fix used throughout this script family).

Everything else -- the FD ghost-extension machinery, stencil orders and
coefficients, network architecture, optimizer/schedule, training loop
structure, output artifact schema, and plotting -- is UNCHANGED from the
SS-SS-SS-SS FDM-PINN hydrostatic script. See that script's docstring (or
fdm_pinn_sinusoidal_hardBC.py's) for the FD-specific caveats (O(h^4)
truncation error, ghost-grid cost, fixed uniform grid requirement); they
all still apply here unchanged.

Outputs (written to ./results/<MODEL_NAME>_<LOADING_CASE>_<BC_CASE>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python fdm_pinn_hydrostatic_hardBC_FFFF.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "FDM_PINN_HardBC"   # >>> VARIANT POINT: change per architecture
LOADING_CASE = "hydrostatic"
BC_CASE = "FFFF"   # all four edges clamped/fixed (w=0 AND dw/dn=0 on every edge)

# Keyed on MODEL_NAME, LOADING_CASE, *and* BC_CASE -- see docstring note.
OUT_DIR = os.path.join("results", f"{MODEL_NAME}_{LOADING_CASE}_{BC_CASE}")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# Rayleigh-Ritz truncation for the clamped reference solution. Both m and n
# run over ALL integers (the clamped admissible functions phi_i =
# 1-cos(2*i*pi*xi) have no odd/even restriction, unlike the SS Navier
# series). See docstring -- cross-validated against half of the
# uniform-FFFF center deflection at this truncation.
RITZ_M_TERMS = 18
RITZ_N_TERMS = 18

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

dx = a_nd / (nx - 1)
dy = b_nd / (ny - 1)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept for the diagnostic bc_val_loss now -------------
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices, now used for the SLOPE diagnostic (bc_slope_loss)
# and the edge-moment diagnostic (NOT a loss term for the clamped case --
# see docstring) -------------------------------------------------------
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # dw/dx = 0 here
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # dw/dy = 0 here

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# GHOST-EXTENDED GRID (for finite-difference stencils) -- UNCHANGED from
# the SS-SS-SS-SS FDM-PINN script.
# ============================================================
GHOST = 3

x_ext_1d = (np.arange(-GHOST, nx + GHOST)) * dx   # length nx + 2*GHOST
y_ext_1d = (np.arange(-GHOST, ny + GHOST)) * dy   # length ny + 2*GHOST
X_ext, Y_ext = np.meshgrid(x_ext_1d, y_ext_1d)    # shape (ny+2G, nx+2G)

X_ext_flat = torch.tensor(X_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
Y_ext_flat = torch.tensor(Y_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
EXT_NY = ny + 2 * GHOST
EXT_NX = nx + 2 * GHOST

# Physical (non-dimensional) grid, as a torch tensor, for evaluating the load
X_nd_t = torch.tensor(X_nd, dtype=DTYPE, device=DEVICE)
Y_nd_t = torch.tensor(Y_nd, dtype=DTYPE, device=DEVICE)

# --- 4th-order-accurate central finite-difference coefficients (Fornberg) --
C1 = torch.tensor([1 / 12, -8 / 12, 0.0, 8 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
C2 = torch.tensor([-1 / 12, 16 / 12, -30 / 12, 16 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
C4 = torch.tensor([-1 / 6, 2.0, -13 / 2, 28 / 3, -13 / 2, 2.0, -1 / 6], dtype=DTYPE, device=DEVICE)


def _apply_stencil_axis(arr, coeffs, axis, ghost, out_len):
    """Apply a centered 1D stencil (`coeffs`, offsets -half..+half) along
    `axis` of `arr`, where `arr` has `ghost` extra points on each side of
    that axis beyond the first `out_len` physical points. Returns a tensor
    with `out_len` along `axis` and all other dimensions unchanged."""
    half = (coeffs.shape[0] - 1) // 2
    assert ghost >= half, "not enough ghost padding for this stencil"
    out = 0.0
    for k in range(-half, half + 1):
        c = coeffs[k + half]
        start = ghost + k
        if axis == 0:
            sl = arr[start:start + out_len, ...]
        else:
            sl = arr[:, start:start + out_len]
        out = out + c * sl
    return out


def d1_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C1, axis, ghost, out_len) / h


def d2_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C2, axis, ghost, out_len) / (h ** 2)


def d4_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C4, axis, ghost, out_len) / (h ** 4)


def fd_derivatives(model):
    """Single forward pass on the ghost-extended grid, then every
    derivative needed for training and post-processing as finite
    differences of that one pass. All returned tensors have shape
    (ny, nx), matching X_nd/Y_nd's row=y, col=x convention."""
    w_ext = model(X_ext_flat, Y_ext_flat).reshape(EXT_NY, EXT_NX)

    w_rows = w_ext[GHOST:GHOST + ny, :]                              # (ny, nx+2G)
    w_xxxx = d4_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)   # (ny, nx)
    w_xx = d2_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)     # (ny, nx)

    w_cols = w_ext[:, GHOST:GHOST + nx]                               # (ny+2G, nx)
    w_yyyy = d4_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx)
    w_yy = d2_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)     # (ny, nx)

    w_yy_full = d2_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)  # (ny, nx+2G)
    w_xxyy = d2_axis(w_yy_full, axis=1, ghost=GHOST, out_len=nx, h=dx)  # (ny, nx)

    w_y_full = d1_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx+2G)
    w_xy = d1_axis(w_y_full, axis=1, ghost=GHOST, out_len=nx, h=dx)    # (ny, nx)

    # First-derivative fields at the PHYSICAL grid, needed for the clamped
    # slope diagnostic (bc_slope_loss) -- new relative to the SS script.
    w_x_full = d1_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)   # (ny, nx)
    w_y_phys = d1_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx)

    w_phys = w_ext[GHOST:GHOST + ny, GHOST:GHOST + nx]                 # (ny, nx)

    return w_phys, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy, w_x_full, w_y_phys


# ============================================================
# ANALYTICAL / REFERENCE SOLUTION -- RAYLEIGH-RITZ SERIES for a fully
# clamped (FFFF) rectangular plate under HYDROSTATIC load
# q(x,y) = q0*(x/a). See module docstring for the derivation and the
# exact-half cross-check against the uniform-FFFF script.
# ============================================================
def _phi(i, xi):
    return 1.0 - np.cos(2.0 * i * np.pi * xi)


def _dphi(i, xi):
    return 2.0 * i * np.pi * np.sin(2.0 * i * np.pi * xi)


def _d2phi(i, xi):
    return 4.0 * (i ** 2) * (np.pi ** 2) * np.cos(2.0 * i * np.pi * xi)


def _solve_ritz_coefficients(M=RITZ_M_TERMS, N=RITZ_N_TERMS):
    """Assemble and solve the Rayleigh-Ritz linear system K @ c = F for the
    clamped-plate deflection coefficients c_ij, for the hydrostatic load."""
    idx_i = np.arange(1, M + 1)
    idx_j = np.arange(1, N + 1)

    Bx = np.ones((M, M)) + 0.5 * np.eye(M)          # Int phi_i phi_k dxi
    By = np.ones((N, N)) + 0.5 * np.eye(N)
    Cx = np.diag(2.0 * np.pi ** 2 * idx_i ** 2)      # Int phi_i' phi_k' dxi
    Cy = np.diag(2.0 * np.pi ** 2 * idx_j ** 2)
    D0x = -Cx                                        # Int phi_i'' phi_k dxi
    D0y = -Cy
    Ax = np.diag(8.0 * np.pi ** 4 * idx_i ** 4)      # Int phi_i'' phi_k'' dxi
    Ay = np.diag(8.0 * np.pi ** 4 * idx_j ** 4)

    K = (np.kron(Ax, By) + np.kron(Bx, Ay)
         + 2.0 * nu_nd * np.kron(D0x, D0y)
         + 2.0 * (1.0 - nu_nd) * np.kron(Cx, Cy)) * D_nd

    # F_mn = q0 * I_m^x * I_n^y, with I_i^x = Int xi*phi_i dxi = 0.5 (ramp,
    # ALL i) and I_j^y = Int 1*phi_j deta = 1 (uniform-in-y, ALL j)
    # -> F is CONSTANT = 0.5*q0 everywhere (see docstring).
    F = 0.5 * q0_nd * np.ones(M * N)

    c_flat = np.linalg.solve(K, F)
    return c_flat.reshape(M, N)


def _ritz_fields(x, y, c, M=RITZ_M_TERMS, N=RITZ_N_TERMS):
    """Reconstruct (w, Mx, My, Mxy) from Ritz coefficients c at arbitrary
    (x, y) numpy arrays (any broadcastable shape)."""
    w = np.zeros_like(x, dtype=float)
    wxx = np.zeros_like(x, dtype=float)
    wyy = np.zeros_like(x, dtype=float)
    wxy = np.zeros_like(x, dtype=float)

    xi = x / a_nd
    eta = y / b_nd

    for i in range(1, M + 1):
        Xi = _phi(i, xi)
        Xi_p = _dphi(i, xi) / a_nd
        Xi_pp = _d2phi(i, xi) / a_nd ** 2
        row = c[i - 1]
        for j in range(1, N + 1):
            cij = row[j - 1]
            if cij == 0.0:
                continue
            Yj = _phi(j, eta)
            Yj_p = _dphi(j, eta) / b_nd
            Yj_pp = _d2phi(j, eta) / b_nd ** 2

            w += cij * Xi * Yj
            wxx += cij * Xi_pp * Yj
            wyy += cij * Xi * Yj_pp
            wxy += cij * Xi_p * Yj_p

    Mx = -D_nd * (wxx + nu_nd * wyy)
    My = -D_nd * (wyy + nu_nd * wxx)
    Mxy = -D_nd * (1.0 - nu_nd) * wxy
    return w, Mx, My, Mxy


_RITZ_C = _solve_ritz_coefficients(RITZ_M_TERMS, RITZ_N_TERMS)


def _ritz_hydrostatic_fields(x, y):
    return _ritz_fields(x, y, _RITZ_C, RITZ_M_TERMS, RITZ_N_TERMS)


def analytical_w_nd(x, y):
    w, _, _, _ = _ritz_hydrostatic_fields(x, y)
    return w


def analytical_Mx_nd(x, y):
    _, Mx, _, _ = _ritz_hydrostatic_fields(x, y)
    return Mx


def analytical_My_nd(x, y):
    _, _, My, _ = _ritz_hydrostatic_fields(x, y)
    return My


def analytical_Mxy_nd(x, y):
    _, _, _, Mxy = _ritz_hydrostatic_fields(x, y)
    return Mxy


def _check_ritz_convergence():
    check_points = {
        "center (0.5, 0.5)": (a_nd / 2, b_nd / 2),
        "off-center (0.75, 0.5)": (0.75 * a_nd, b_nd / 2),
    }
    M2, N2 = RITZ_M_TERMS + 6, RITZ_N_TERMS + 6
    c_richer = _solve_ritz_coefficients(M2, N2)
    for label, (xc, yc) in check_points.items():
        x_c = np.array([[xc]])
        y_c = np.array([[yc]])
        w1, _, _, _ = _ritz_fields(x_c, y_c, _RITZ_C, RITZ_M_TERMS, RITZ_N_TERMS)
        w2, _, _, _ = _ritz_fields(x_c, y_c, c_richer, M2, N2)
        rel_diff = abs(w1[0, 0] - w2[0, 0]) / (abs(w2[0, 0]) + 1e-30)
        print(f"[Ritz series check] w at {label}: relative change from "
              f"(M={RITZ_M_TERMS},N={RITZ_N_TERMS}) -> (M={M2},N={N2}): {rel_diff:.3e}")
        if rel_diff > 1e-4:
            print("  WARNING: series may not be fully converged at this point; "
                  "consider increasing RITZ_M_TERMS / RITZ_N_TERMS. (This check "
                  "does not cover moments or corner regions.)")
    edge_x_pts = np.array([[0.0, 1.0]])
    edge_y_pts = np.array([[0.5, 0.5]])
    w_e, _, _, _ = _ritz_fields(edge_x_pts, edge_y_pts, _RITZ_C, RITZ_M_TERMS, RITZ_N_TERMS)
    print(f"[Ritz BC check] w at x=0,1 (y=0.5): {w_e.ravel()} (should be ~0)")
    # Exact-half cross-check against a fresh uniform-load Ritz solve on
    # the SAME clamped basis/K (see docstring derivation) -- an
    # independent internal consistency check, since no external textbook
    # benchmark for clamped+hydrostatic was located.
    F_uniform = q0_nd * np.ones(RITZ_M_TERMS * RITZ_N_TERMS)
    idx_i = np.arange(1, RITZ_M_TERMS + 1)
    idx_j = np.arange(1, RITZ_N_TERMS + 1)
    Bx = np.ones((RITZ_M_TERMS, RITZ_M_TERMS)) + 0.5 * np.eye(RITZ_M_TERMS)
    By = np.ones((RITZ_N_TERMS, RITZ_N_TERMS)) + 0.5 * np.eye(RITZ_N_TERMS)
    Cx = np.diag(2.0 * np.pi ** 2 * idx_i ** 2)
    Cy = np.diag(2.0 * np.pi ** 2 * idx_j ** 2)
    Ax = np.diag(8.0 * np.pi ** 4 * idx_i ** 4)
    Ay = np.diag(8.0 * np.pi ** 4 * idx_j ** 4)
    K = (np.kron(Ax, By) + np.kron(Bx, Ay) + 2.0 * nu_nd * np.kron(-Cx, -Cy)
         + 2.0 * (1.0 - nu_nd) * np.kron(Cx, Cy)) * D_nd
    c_uniform = np.linalg.solve(K, F_uniform).reshape(RITZ_M_TERMS, RITZ_N_TERMS)
    w_uniform_center, _, _, _ = _ritz_fields(np.array([[0.5]]), np.array([[0.5]]),
                                              c_uniform, RITZ_M_TERMS, RITZ_N_TERMS)
    w_hydro_center, _, _, _ = _ritz_fields(np.array([[0.5]]), np.array([[0.5]]),
                                            _RITZ_C, RITZ_M_TERMS, RITZ_N_TERMS)
    ratio = w_hydro_center[0, 0] / w_uniform_center[0, 0]
    print(f"[Ritz exact-half check] hydrostatic/uniform center-deflection ratio = "
          f"{ratio:.8f} (must equal exactly 0.5 -- linear-superposition + "
          f"x->a-x BC symmetry argument, BC-independent, see docstring)")


_check_ritz_convergence()


def hydrostatic_load_nd(x, y):
    return q0_nd * (x / a_nd)


Q_grid = hydrostatic_load_nd(X_nd_t, Y_nd_t)   # (ny, nx), precomputed once -- ramps 0 -> q0_nd across x


# ============================================================
# MODEL   (>>> VARIANT POINT: swap the input block for FF-PINN /
#          swap the residual block for FDM-PINN / FF+FDM-PINN)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)   # >>> VARIANT POINT: Fourier embed here for FF variants
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained CLAMPED BC: both w=0 AND dw/dn=0 on every edge.
        # Squaring each factor zeros the function AND its derivative at
        # xi=0,1 and eta=0,1. Evaluated fine for ghost points outside
        # [0,a]x[0,b] too -- smooth polynomial continuation, no special-
        # casing needed for the FD stencils.
        xi = x / a_nd
        eta = y / b_nd
        envelope = (xi * (1 - xi)) ** 2 * (eta * (1 - eta)) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via finite differences, not autograd)
# ============================================================
def compute_loss():
    w_grid, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy, w_x_full, w_y_phys = fd_derivatives(model)

    biharmonic = w_xxxx + 2 * w_xxyy + w_yyyy
    residual = (biharmonic - Q_grid).reshape(-1, 1)
    pde_loss = torch.mean(residual[int_idx] ** 2)

    w_flat = w_grid.reshape(-1, 1)
    # DIAGNOSTICS ONLY -- both should be ~0 exactly, by construction of the
    # squared clamped envelope (value AND normal slope vanish on all four
    # edges). Neither is part of the training loss.
    bc_val_loss = torch.mean(w_flat[bc_idx] ** 2)
    wx_flat = w_x_full.reshape(-1, 1)
    wy_flat = w_y_phys.reshape(-1, 1)
    bc_slope_loss = torch.mean(wx_flat[edge_x_idx] ** 2) + torch.mean(wy_flat[edge_y_idx] ** 2)

    # DIAGNOSTIC ONLY -- for a CLAMPED edge, Mx / My are generally NONZERO
    # (the edge moment is what resists rotation), so this is logged for
    # reference/comparison across scripts only, never used as a loss term.
    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mx_flat = Mx.reshape(-1, 1)
    My_flat = My.reshape(-1, 1)
    edge_moment_diag = torch.mean(Mx_flat[edge_x_idx] ** 2) + torch.mean(My_flat[edge_y_idx] ** 2)

    total = pde_loss
    return total, pde_loss, bc_val_loss, bc_slope_loss, edge_moment_diag, w_flat


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history = [], [], []
bc_val_history, bc_slope_history, moment_diag_history = [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc_val, bc_slope, moment_diag, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_val_history.append(bc_val.item())
    bc_slope_history.append(bc_slope.item())
    moment_diag_history.append(moment_diag.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss(pde) {loss_val:.6e} "
              f"(hardBC_val {bc_val.item():.3e}, hardBC_slope {bc_slope.item():.3e}, "
              f"edge_moment_diag {moment_diag.item():.3e}) "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_val_residual_diagnostic": bc_val_history,
    "hard_bc_slope_residual_diagnostic": bc_slope_history,
    "edge_moment_diagnostic": moment_diag_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde = pde_history[-1]
final_bc_val = bc_val_history[-1]
final_bc_slope = bc_slope_history[-1]
final_moment_diag = moment_diag_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "final_pde_loss": final_pde,
    "final_hard_bc_val_residual_diagnostic": final_bc_val,
    "final_hard_bc_slope_residual_diagnostic": final_bc_slope,
    "final_edge_moment_diagnostic": final_moment_diag,
    "note": "Total training loss is PDE residual only for the clamped (FFFF) "
            "case -- both w=0 and dw/dn=0 are hard-constrained by the squared "
            "envelope, so no soft BC/moment term is needed or appropriate here.",
}
if final_bc_val > 1e-20:
    print(f"WARNING: hard_bc_val_residual_diagnostic = {final_bc_val:.3e} is not "
          f"~0 -- double-check the envelope function / grid point exactness "
          f"(float64 expected).")
if final_bc_slope > 1e-16:
    print(f"WARNING: hard_bc_slope_residual_diagnostic = {final_bc_slope:.3e} is "
          f"not ~0 -- the squared envelope should zero the slope at the edges "
          f"exactly; double-check the FD stencils / grid point exactness.")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "ghost_points_per_side": GHOST,
    "forward_pass_points": EXT_NX * EXT_NY,
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via the same
# finite-difference machinery used in training.
# ============================================================
def compute_full_fields():
    with torch.no_grad():
        w_grid, w_xx, w_yy, _, _, _, w_xy, _, _ = fd_derivatives(model)
        Mx = -(w_xx + nu_nd * w_yy)
        My = -(w_yy + nu_nd * w_xx)
        Mxy = -(1 - nu_nd) * w_xy

    w_grid_np = w_grid.cpu().numpy()
    Mx_np = Mx.cpu().numpy()
    My_np = My.cpu().numpy()
    Mxy_np = Mxy.cpu().numpy()
    return w_grid_np, Mx_np, My_np, Mxy_np


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd, Mx_anal_nd, My_anal_nd, Mxy_anal_nd = _ritz_hydrostatic_fields(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
# NOTE: since the load is not x-symmetric, the plate CENTER is not
# necessarily the point of maximum deflection -- kept here only for
# consistency with the other loading-case scripts (comparable metric
# across scripts), not because it's expected to be the worst-case point.
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "ritz_m_terms": RITZ_M_TERMS,
    "ritz_n_terms": RITZ_N_TERMS,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Reference (Rayleigh-Ritz)")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title(f"Convergence of Center Deflection ({BC_CASE}, {LOADING_CASE})")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total (PDE) Loss")
plt.plot(df_hist["epoch"], df_hist["hard_bc_val_residual_diagnostic"],
         label="Hard-BC value residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["hard_bc_slope_residual_diagnostic"],
         label="Hard-BC slope residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["edge_moment_diagnostic"],
         label="Edge moment (diagnostic, expected NONZERO for clamped edges)", alpha=0.7)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss / diagnostic"); plt.title(f"Training Loss Curve ({BC_CASE})")
plt.legend(fontsize=8); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, f"Reference Plate Deflection (Rayleigh-Ritz, {BC_CASE})", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Reference Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Reference Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Reference Twisting Moment $M_{xy}$ (note: this is a\ntruncated Ritz series -- see convergence caveats)",
             "Mxy (N·m/m)", "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Reference displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Reference vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Reference {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, {BC_CASE} (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

[Ritz series check] w at center (0.5, 0.5): relative change from (M=18,N=18) -> (M=24,N=24): 3.421e-05
[Ritz series check] w at off-center (0.75, 0.5): relative change from (M=18,N=18) -> (M=24,N=24): 4.153e-05
[Ritz BC check] w at x=0,1 (y=0.5): [0. 0.] (should be ~0)
[Ritz exact-half check] hydrostatic/uniform center-deflection ratio = 0.50000000 (must equal exactly 0.5 -- linear-superposition + x->a-x BC symmetry argument, BC-independent, see docstring)
Epoch 0 Loss(pde) 2.542735e-01 (hardBC_val 0.000e+00, hardBC_slope 2.826e-19, edge_moment_diag 6.132e-05) Center 2.768521e-04 (336.92 ms/epoch)
Epoch 100 Loss(pde) 3.552466e-02 (hardBC_val 0.000e+00, hardBC_slope 5.280e-16, edge_moment_diag 5.657e-04) Center 5.991223e-04 (9.79 ms/epoch)
Epoch 200 Loss(pde) 2.784984e-02 (hardBC_val 0.000e+00, hardBC_slope 7.151e-16, edge_moment_diag 5.598e-04) Center 5.987730e-04 (10.85 ms/epoch)
Epoch 300 Loss(pde) 1.908700e-02 (hardBC_val 0.000e+00, hardBC_slope 8.516e-16, edge_moment_diag 5.855e-04

C:\Users\vivek\AppData\Local\Temp\ipykernel_4512\2433770274.py:627: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'FDM_PINN_HardBC', 'loading_case': 'hydrostatic', 'bc_case': 'FFFF', 'final_pde_loss': 0.000540445120532095, 'final_hard_bc_val_residual_diagnostic': 0.0, 'final_hard_bc_slope_residual_diagnostic': 2.0602530758581394e-13, 'final_edge_moment_diagnostic': 0.0006109478747055115, 'note': 'Total training loss is PDE residual only for the clamped (FFFF) case -- both w=0 and dw/dn=0 are hard-constrained by the squared envelope, so no soft BC/moment term is needed or appropriate here.'}
Saved cost_metrics.json: {'model': 'FDM_PINN_HardBC', 'loading_case': 'hydrostatic', 'bc_case': 'FFFF', 'device': 'cuda', 'epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'mean_time_per_epoch_s': 0.011037237132992596, 'median_time_per_epoch_s': 0.010897899977862835, 'total_training_time_s': 186.7985561999958, 'peak_gpu_memory_mb': 29.949952, 'n_trainable_params': 8578, 'ghost_points_per_side': 3, 'forward_pass_p